# بارگذاری پایگاه داده در سایت (نسخه‌ی 12)

فقط یک سلول: روی ▶️ بزنید و اجازه‌ی دسترسی به Google Drive را بدهید. کارت گرافیک لازم نیست.
پایگاه داده‌ای که نوت‌بوک اصلی (سلول ۵) در `matrag_data` ساخته، با همان بردارها در Qdrant بارگذاری می‌شود
و بعد جست‌وجوی سایت با همان سؤال‌های ارزیابی سنجیده می‌شود. اگر قطع شد دوباره اجرا کنید؛ از همان‌جا ادامه می‌دهد.

In [ ]:
#@title ▶️ بارگذاری در سایت

import os
# Keep Colab's preinstalled JAX/TensorFlow from reserving the GPU memory
# (otherwise the embedding model and Ollama run out of memory).
os.environ.update({'JAX_PLATFORMS': 'cpu', 'XLA_PYTHON_CLIENT_PREALLOCATE': 'false',
                   'TF_FORCE_GPU_ALLOW_GROWTH': 'true', 'USE_TF': '0', 'USE_FLAX': '0',
                   'TF_CPP_MIN_LOG_LEVEL': '3'})
from google.colab import drive
drive.mount('/content/drive')

WORK = '/content/drive/MyDrive/matrag_data'
os.environ['MATRAG_DATA_DIR'] = f'{WORK}/data'
os.environ['MATRAG_STORAGE_DIR'] = '/content/storage'
os.environ['MATRAG_CORPUS'] = 'rag-optics'
print('Copying the database from Drive...')
!mkdir -p /content/storage && cp -rT "{WORK}/storage" /content/storage

BRANCH = "claude/festive-bardeen-nnw123"  #@param {type:"string"}
# Leave the project folder before deleting it (re-running this cell used to fail
# with "Unable to read current working directory").
%cd /content
!rm -rf /content/RAG && git clone -q --depth 1 -b {BRANCH} https://github.com/aminsadidi/RAG.git /content/RAG
%cd /content/RAG
!pip install -q -e . 2>&1 | grep -iE "^error" || true
# An editable install is only seen by Python after a restart; adding the source
# folder to the path makes it importable in this session right away.
import sys
if '/content/RAG/src' not in sys.path:
    sys.path.insert(0, '/content/RAG/src')

# The Qdrant address and key: matrag_data/qdrant.env on Drive (KEY=value lines), else Colab Secrets.
import os
KEYS_FILE = f'{WORK}/qdrant.env'
if os.path.exists(KEYS_FILE):
    for line in open(KEYS_FILE, encoding='utf-8'):
        key, sep, value = line.strip().partition('=')
        if sep and not key.startswith('#'):
            os.environ[key.strip()] = value.strip().strip('"')
else:
    from google.colab import userdata
    for key in ('QDRANT_URL', 'QDRANT_API_KEY'):
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            pass
assert os.environ.get('QDRANT_URL') and os.environ.get('QDRANT_API_KEY'), f'Qdrant key not found ({KEYS_FILE})'

%cd {WORK}
!matrag --corpus rag-optics export-qdrant 2>&1 | grep -vE "Warning|Loading"
if os.path.exists(f'{WORK}/data/rag-optics/gold/questions.csv'):
    !matrag --corpus rag-optics evaluate retrieval --backend qdrant 2>&1 | grep -vE "Warning|Loading" | tee -a "{WORK}/logs/eval.log"
%cd /content/RAG
print('✅ Done: https://matrag.ethanjamescarter1995.workers.dev')